In [2]:
import pandas as pd

In [4]:
PATH = 'data/earning_call_presentations/clean_tr_sp500_final_2000s.parquet'

In [12]:
df = pd.read_parquet(PATH)
df = df.sort_values(by = ['transcriptid', 'componentorder'])

transcripts1 = df.groupby('transcriptid')['transcript_text'].agg(lambda segs: ' '.join(segs)).reset_index(name='full_transcript')

In [16]:
PATH = 'data/earning_call_presentations/clean_tr_sp500_final_2010_2015.parquet'
df = pd.read_parquet(PATH)
df = df.sort_values(by = ['transcriptid', 'componentorder'])

In [18]:
transcripts2 = df.groupby('transcriptid')['transcript_text'].agg(lambda segs: ' '.join(segs)).reset_index(name='full_transcript')

In [44]:
transcripts = pd.concat([transcripts1, transcripts2])[['full_transcript']].rename(columns = {'full_transcript' : 'text'})

In [48]:
transcripts.to_csv('training.csv', index = False)

In [52]:
df = pd.read_csv('training.csv')

In [64]:
df

,text
0,"Good afternoon. My name is Karen, and I'll be ..."
1,"Greetings, and welcome to the First Quarter Fi..."
2,"Ladies and gentlemen, thank you for standing b..."
3,"Good morning. My name is Joshua, and I will be..."
4,Welcome to the Bed Bath & Beyond's Third Quart...
...,...
19427,"Good afternoon, ladies and gentlemen, and welc..."
19428,Welcome to Franklin Resources Earnings Comment...
19429,Welcome to Cisco Systems Fourth Quarter and Fi...
19430,Good afternoon. My name is Marvin and I will b...


In [2]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

In [4]:
df = pd.read_csv("processed_text.csv")        
ds = Dataset.from_pandas(df, preserve_index=False)

split = ds.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split["train"], split["test"]

MODEL_NAME = "alikLab/NoLBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

def tokenize_full(batch):
    return tokenizer(
        batch["text"],
        truncation =True,             
        padding="max_length",         
        max_length=512,
        return_special_tokens_mask=True,
    )

train_tok = train_ds.map(
    tokenize_full,
    batched=True,
    remove_columns=["text"]
)
val_tok = val_ds.map(
    tokenize_full,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir="nolbert-finetuned",
    num_train_epochs=3.0,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=5000,

    logging_strategy="steps",
    logging_steps=1000,

    save_strategy="steps",
    save_steps=5000,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("nolbert-earned-calls")


Map:   0%|          | 0/249469 [00:00<?, ? examples/s]

Map:   0%|          | 0/13130 [00:00<?, ? examples/s]

/Users/kevinsu/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 

In [36]:
df = pd.read_csv('training.csv')

In [46]:
df

,text
0,"Good afternoon. My name is Karen, and I'll be ..."
1,"Greetings, and welcome to the First Quarter Fi..."
2,"Ladies and gentlemen, thank you for standing b..."
3,"Good morning. My name is Joshua, and I will be..."
4,Welcome to the Bed Bath & Beyond's Third Quart...
...,...
19427,"Good afternoon, ladies and gentlemen, and welc..."
19428,Welcome to Franklin Resources Earnings Comment...
19429,Welcome to Cisco Systems Fourth Quarter and Fi...
19430,Good afternoon. My name is Marvin and I will b...


In [48]:
df_30 = df.sample(frac = 0.3, random_state = 42)

In [52]:
df_30.to_csv('training_30.csv', index = False)

In [80]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

def chunk_text(text, max_words=250):
    sents = sent_tokenize(text)
    chunks = []
    current = []
    count = 0
    
    for sent in sents:
        w = word_tokenize(sent)
        if count + len(w) > max_words:
            chunks.append(" ".join(current))
            current = [sent]
            count = len(w)
        else:
            current.append(sent)
            count += len(w)
    if current:
        chunks.append(" ".join(current))
    return chunks

In [82]:
df_30["chunks"] = df_30["text"].apply(chunk_text)

df_chunk = df_30.explode("chunks").reset_index(drop=True)

df_chunk = df_chunk.drop(columns = ["text"]).rename(columns={"chunks":"text"})

In [83]:
df_chunk['text'].iloc[1]

"Before beginning, I would like to also remind you that we will be hosting our Investor Day on November 16 at Lincoln Financial Field in Philadelphia. I trust you received our invitation, and we hope most of you will be able to attend. Presenting on today's call are Dennis Glass, President and Chief Executive Officer; and Randy Freitag, Chief Financial Officer. After their prepared remarks, we will move to the question-and-answer portion of the call. I would now like to turn the call over to Dennis. Thank you, Chris. Good morning, everyone. 2017 continues to be an outstanding year as third quarter operating earnings and earnings per share were a record. Operating EPS increased 7% compared to the prior year and marked the first time quarterly EPS has exceeded $2 per share. Book value per share increased 8% and ROE was 13.6%. As you know, during the quarter, we also completed our comprehensive annual assumption review, which had a small impact on our financials. Randy will provide you mo

In [86]:
df_chunk.to_csv('processed_text.csv', index = False)